# Week 85, Day 6: Exam debrief

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w085d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('sales.csv', 'w', encoding='utf-8') as f:
    f.write('date,product,qty\n2026-01-05,P1,3\n2026-01-09,P2,1\n2026-01-17,P3,10\n2026-01-28,P1,1\n2026-02-02,P4,2\n2026-02-11,P2,2\n2026-02-14,P3,4\n2026-02-20,P5,1\n2026-02-27,P1,2\n')
with open('products.csv', 'w', encoding='utf-8') as f:
    f.write('product,name,category,price\nP1,Notebook,stationery,2.50\nP2,Desk lamp,home,24.99\nP3,Pens,stationery,1.20\nP4,Mug,home,6.75\nP5,Headphones,electronics,59.00\n')

In [ ]:
def _v(text):
    return tuple(int(p) for p in text.strip().split("."))

def _ok(version, clause):
    clause = clause.strip()
    if clause.startswith("^"):
        low = _v(clause[1:])
        return low <= version < (low[0] + 1, 0, 0)
    for op in (">=", "==", "<"):
        if clause.startswith(op):
            other = _v(clause[len(op):])
            return {">=": version >= other, "==": version == other, "<": version < other}[op]
    raise ValueError(f"bad clause {clause}")

def resolve(available, requirements):
    specs = {}
    for name, spec in requirements:
        specs.setdefault(name, []).extend(spec.split(","))
    result = {}
    for name, clauses in specs.items():
        good = [v for v in available.get(name, []) if all(_ok(_v(v), c) for c in clauses)]
        if not good:
            raise ValueError(f"no version of {name}")
        result[name] = max(good, key=_v)
    return result

print(resolve({"requests": ["2.9.0", "2.31.0", "2.10.0", "3.0.0"]}, [("requests", "^2.9.0"), ("requests", "<2.20.0")]))

In [ ]:
import sqlite3

def make_db():
    conn = sqlite3.connect(":memory:")
    conn.executescript('''
    CREATE TABLE events (id INTEGER PRIMARY KEY, title TEXT, capacity INTEGER);
    CREATE TABLE signups (id INTEGER PRIMARY KEY, event_id INTEGER, person TEXT, status TEXT);
    INSERT INTO events VALUES (1, 'Pottery', 2), (2, 'Coding club', 1), (3, 'Book group', 5);
    ''')
    return conn

def sign_up(conn, event_id, person):
    with conn:
        if conn.execute("SELECT 1 FROM signups WHERE event_id = ? AND person = ?", (event_id, person)).fetchone():
            raise ValueError("already signed up")
        (capacity,) = conn.execute("SELECT capacity FROM events WHERE id = ?", (event_id,)).fetchone()
        (booked,) = conn.execute("SELECT COUNT(*) FROM signups WHERE event_id = ? AND status = 'booked'", (event_id,)).fetchone()
        status = "booked" if booked < capacity else "waiting"
        conn.execute("INSERT INTO signups (event_id, person, status) VALUES (?, ?, ?)", (event_id, person, status))
    return status

def cancel(conn, event_id, person):
    with conn:
        row = conn.execute("SELECT id, status FROM signups WHERE event_id = ? AND person = ?", (event_id, person)).fetchone()
        if row is None:
            raise ValueError("not signed up")
        conn.execute("DELETE FROM signups WHERE id = ?", (row[0],))
        if row[1] != "booked":
            return None
        nxt = conn.execute("SELECT id, person FROM signups WHERE event_id = ? AND status = 'waiting' ORDER BY id LIMIT 1",
                           (event_id,)).fetchone()
        if nxt is None:
            return None
        conn.execute("UPDATE signups SET status = 'booked' WHERE id = ?", (nxt[0],))
        return nxt[1]

def summary(conn):
    return conn.execute('''
        SELECT e.title,
               COALESCE(SUM(s.status = 'booked'), 0),
               COALESCE(SUM(s.status = 'waiting'), 0)
        FROM events e LEFT JOIN signups s ON s.event_id = e.id
        GROUP BY e.id ORDER BY e.title
    ''').fetchall()

conn = make_db()
print([sign_up(conn, 1, p) for p in ("ada", "bo", "cy")], cancel(conn, 1, "ada"), summary(conn))

In [ ]:
import hashlib

def respond(method, path, headers, pages):
    if path not in pages:
        return 404, {}, "not found"
    if method not in ("GET", "HEAD"):
        return 405, {"Allow": "GET, HEAD"}, ""
    body = pages[path]
    data = body.encode("utf-8")
    etag = '"' + hashlib.sha256(data).hexdigest()[:16] + '"'
    lowered = {k.lower(): v for k, v in headers.items()}
    if lowered.get("if-none-match") == etag:
        return 304, {"ETag": etag}, ""
    out = {"ETag": etag, "Content-Type": "text/plain; charset=utf-8", "Content-Length": str(len(data))}
    return 200, out, "" if method == "HEAD" else body

pages = {"/": "Welcome"}
status, headers, body = respond("GET", "/", {}, pages)
print(status, respond("GET", "/", {"IF-NONE-MATCH": headers["ETag"]}, pages))

In [ ]:
import hashlib
import hmac

class ResetTokens:
    def __init__(self, clock, make_token, ttl=900):
        self.clock, self.make_token, self.ttl = clock, make_token, ttl
        self.by_user = {}

    @staticmethod
    def _digest(token):
        return hashlib.sha256(token.encode()).hexdigest()

    def issue(self, user):
        token = self.make_token()
        self.by_user[user] = (self._digest(token), self.clock() + self.ttl)
        return token

    def redeem(self, token):
        digest = self._digest(token)
        for user, (stored, expiry) in list(self.by_user.items()):
            if hmac.compare_digest(stored, digest):
                del self.by_user[user]
                if self.clock() >= expiry:
                    break
                return user
        raise ValueError("invalid token")

import itertools
counter = itertools.count()
rt = ResetTokens(lambda: 0, lambda: f"token-{next(counter)}")
t = rt.issue("ada")
print(rt.redeem(t), rt.by_user)

In [ ]:
import pandas as pd

def _joined(sales, products):
    df = sales.merge(products, on="product", how="left", validate="many_to_one")
    return df.assign(revenue=df["qty"] * df["price"], month=df["date"].str[:7])

def revenue_table(sales, products):
    df = _joined(sales, products)
    pivot = df.pivot_table(index="month", columns="category", values="revenue", aggfunc="sum", fill_value=0)
    return pivot.round(2).to_dict("index")

def best_sellers(sales, products):
    totals = _joined(sales, products).groupby(["category", "name"])["revenue"].sum().reset_index()
    top = totals.loc[totals.groupby("category")["revenue"].idxmax()]
    return dict(zip(top["category"], top["name"]))

import pandas as pd
sales, products = pd.read_csv("sales.csv"), pd.read_csv("products.csv")
print(revenue_table(sales, products))
print(best_sellers(sales, products))

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def _folds(n, k):
    sizes = [n // k + (1 if i < n % k else 0) for i in range(k)]
    start = 0
    for size in sizes:
        test = np.arange(start, start + size)
        train = np.concatenate([np.arange(0, start), np.arange(start + size, n)])
        yield train, test
        start += size

def _model(k):
    return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))

def choose_k(X, y, ks, folds=5):
    scores = {}
    for k in ks:
        results = [_model(k).fit(X[tr], y[tr]).score(X[te], y[te]) for tr, te in _folds(len(X), folds)]
        scores[k] = round(float(np.mean(results)), 3)
    best = max(scores.values())
    return min(k for k in scores if scores[k] == best), scores

def evaluate(X_train, y_train, X_test, y_test, k):
    pred = _model(k).fit(X_train, y_train).predict(X_test)
    tp = int(((pred == 1) & (y_test == 1)).sum())
    flagged, actual = int((pred == 1).sum()), int((y_test == 1).sum())
    return {"accuracy": round(float((pred == y_test).mean()), 3),
            "precision": round(tp / flagged, 3) if flagged else 0.0,
            "recall": round(tp / actual, 3) if actual else 0.0}

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
X, y = make_classification(n_samples=240, n_features=6, n_informative=3, weights=[0.7], random_state=11)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
best, scores = choose_k(X_train, y_train, [1, 3, 5, 9, 15])
print(best, scores)
print(evaluate(X_train, y_train, X_test, y_test, best))

In [ ]:
import hmac
import json
import math
import sqlite3
from urllib.parse import parse_qs, urlsplit

def _p95(values):
    ordered = sorted(values)
    return ordered[math.ceil(0.95 * len(ordered)) - 1]

def _valid(event):
    if not isinstance(event, dict):
        return False
    endpoint, status, ms = event.get("endpoint"), event.get("status"), event.get("ms")
    return (isinstance(endpoint, str) and endpoint.startswith("/")
            and isinstance(status, int) and not isinstance(status, bool) and 100 <= status <= 599
            and isinstance(ms, (int, float)) and not isinstance(ms, bool) and ms >= 0)

class Metrics:
    ROUTES = {"/events": "POST", "/summary": "GET", "/slo": "GET"}

    def __init__(self, api_keys, clock):
        self.api_keys, self.clock = api_keys, clock
        self.conn = sqlite3.connect(":memory:")
        self.conn.execute("CREATE TABLE events (service TEXT, endpoint TEXT, status INTEGER, ms REAL, at REAL)")

    def _service(self, headers):
        given = {k.lower(): v for k, v in headers.items()}.get("x-api-key", "")
        found = None
        for key, service in self.api_keys.items():
            if hmac.compare_digest(given.encode(), key.encode()):
                found = service
        return found

    def handle(self, method, path, headers, body):
        service = self._service(headers)
        if service is None:
            return 401, {"error": "bad key"}
        parts = urlsplit(path)
        query = {k: v[0] for k, v in parse_qs(parts.query).items()}
        if parts.path not in self.ROUTES:
            return 404, {"error": "not found"}
        if method != self.ROUTES[parts.path]:
            return 405, {"error": "method not allowed"}
        if parts.path == "/events":
            return self._ingest(service, body)
        if parts.path == "/summary":
            return self._summary(service, query)
        return self._slo(service, query)

    def _ingest(self, service, body):
        try:
            events = json.loads(body)
        except ValueError:
            return 400, {"error": "bad request"}
        if not isinstance(events, list):
            return 400, {"error": "bad request"}
        for i, event in enumerate(events):
            if not _valid(event):
                return 400, {"error": f"bad event {i}"}
        now = self.clock()
        with self.conn:
            self.conn.executemany("INSERT INTO events VALUES (?, ?, ?, ?, ?)",
                                  [(service, e["endpoint"], e["status"], float(e["ms"]), now) for e in events])
        return 202, {"accepted": len(events)}

    def _rows(self, service, window, endpoint=None):
        sql = "SELECT endpoint, status, ms FROM events WHERE service = ? AND at > ?"
        params = [service, self.clock() - window]
        if endpoint is not None:
            sql += " AND endpoint = ?"
            params.append(endpoint)
        return self.conn.execute(sql, params).fetchall()

    def _summary(self, service, query):
        rows = self._rows(service, float(query.get("window", 3600)), query.get("endpoint"))
        if not rows:
            return 200, {"count": 0, "error_rate": 0.0, "p95": None, "slowest": None}
        by_endpoint = {}
        for endpoint, status, ms in rows:
            by_endpoint.setdefault(endpoint, []).append(ms)
        p95s = {e: _p95(v) for e, v in by_endpoint.items()}
        slowest = min(p95s, key=lambda e: (-p95s[e], e))
        p95 = _p95([ms for _, _, ms in rows])
        return 200, {"count": len(rows), "error_rate": round(sum(s >= 500 for _, s, _ in rows) / len(rows), 3),
                     "p95": int(p95) if float(p95).is_integer() else p95, "slowest": slowest}

    def _slo(self, service, query):
        target = float(query.get("target", 0.99))
        rows = self._rows(service, 3600)
        errors = sum(s >= 500 for _, s, _ in rows)
        fraction = errors / len(rows) if rows else 0.0
        used = round(fraction / (1 - target), 2)
        status = "ok" if used <= 0.5 else "burning" if used <= 1 else "exhausted"
        return 200, {"availability": round(1 - fraction, 4), "budget_used": used, "status": status}

import json
keys = {"demo-" + "key": "shop"}
m = Metrics(keys, lambda: 100.0)
print(m.handle("POST", "/events", {"X-Api-Key": "demo-key"}, json.dumps([{"endpoint": "/a", "status": 200, "ms": 12}, {"endpoint": "/a", "status": 500, "ms": 340}])))
print(m.handle("GET", "/summary", {"x-api-key": "demo-key"}, ""), m.handle("GET", "/slo?target=0.9", {"x-api-key": "demo-key"}, ""))

### Exercise: Second chance: tests at the boundaries (B1)

`grades.py` has `grade(score)` for whole-number scores from 0 to 100: 70 or more is `"A"`, 60 `"B"`, 50 `"C"`, 40 `"D"`, otherwise `"F"`; scores outside 0 to 100 raise `ValueError`. Write tests that pass and catch all five broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('grades.py', 'w', encoding='utf-8') as f:
    f.write('def grade(score):\n    if score < 0 or score > 100:\n        raise ValueError("score out of range")\n    if score >= 70:\n        return "A"\n    if score >= 60:\n        return "B"\n    if score >= 50:\n        return "C"\n    if score >= 40:\n        return "D"\n    return "F"\n')

In [ ]:
import pytest
from grades import grade

def test_top():
    assert grade(95) == "A"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"grades.py\").read()\nBROKEN = {\n    \"A from 71\": GOOD.replace(\"if score >= 70:\", \"if score > 70:\"),\n    \"D from 41\": GOOD.replace(\"if score >= 40:\", \"if score > 40:\"),\n    \"C from 55\": GOOD.replace(\"if score >= 50:\", \"if score >= 55:\"),\n    \"allows 101\": GOOD.replace(\"score > 100\", \"score > 101\"),\n    \"allows negatives\": GOOD.replace(\"score < 0 or \", \"\"),\n}\ndef test_suite():\n    \"all five caught\"\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"grades\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"grades.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Exercise: Second chance: safe upgrades (B3)

Write `newest_compatible(versions, current)`: the highest version in `versions` that is at least `current` and **compatible** with it (the same MAJOR; for MAJOR 0, the same MINOR too), or `current` itself if none is newer. Then `upgrade_plan(installed, available)`: a dict, sorted by name, from each installed package to its newest compatible version, only for packages where that's newer than what's installed.

In [ ]:
def newest_compatible(versions, current):
    ___

def upgrade_plan(installed, available):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_upgrades():\n    \"same major, as numbers\"\n    assert newest_compatible([\"1.2.0\", \"1.10.3\", \"2.0.0\", \"1.9.9\"], \"1.2.0\") == \"1.10.3\"\n    assert newest_compatible([\"0.3.9\", \"0.4.0\", \"0.3.10\"], \"0.3.1\") == \"0.3.10\"\n    assert newest_compatible([\"1.0.0\"], \"1.4.0\") == \"1.4.0\"\n    plan = upgrade_plan({\"rich\": \"13.0.0\", \"click\": \"8.1.7\", \"tiny\": \"0.1.0\"},\n                        {\"rich\": [\"13.7.1\", \"14.0.0\"], \"click\": [\"8.1.7\", \"9.0.0\"], \"tiny\": [\"0.1.4\", \"0.2.0\"]})\n    assert plan == {\"rich\": \"13.7.1\", \"tiny\": \"0.1.4\"} and list(plan) == [\"rich\", \"tiny\"]")

### Exercise: Second chance: no double bookings (B4)

With the given `make_db()`, write `book(conn, room, start, end, who)` for times in minutes: raise `ValueError("bad times")` unless `start < end`, and `ValueError("clash")` if the room has a booking that overlaps (`start < other_end` and `end > other_start`; touching is fine); otherwise insert it, all in one transaction. Then `free_rooms(conn, start, end)`: the sorted names of rooms with no overlapping booking.

In [ ]:
import sqlite3

def make_db():
    conn = sqlite3.connect(":memory:")
    conn.executescript('''
    CREATE TABLE rooms (name TEXT PRIMARY KEY);
    CREATE TABLE bookings (id INTEGER PRIMARY KEY, room TEXT, start INTEGER, end_ INTEGER, who TEXT);
    INSERT INTO rooms VALUES ('Attic'), ('Boardroom'), ('Cellar');
    ''')
    return conn

def book(conn, room, start, end, who):
    ___

def free_rooms(conn, start, end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rooms():\n    \"overlaps, not touches\"\n    conn = make_db()\n    book(conn, \"Attic\", 540, 600, \"ada\")\n    book(conn, \"Attic\", 600, 660, \"bo\")\n    for args, message in (((\"Attic\", 590, 610, \"cy\"), \"clash\"), ((\"Attic\", 500, 700, \"cy\"), \"clash\"), ((\"Cellar\", 600, 600, \"cy\"), \"bad times\")):\n        try:\n            book(conn, *args)\n        except ValueError as e:\n            assert str(e) == message\n        else:\n            raise AssertionError(args)\n    assert conn.execute(\"SELECT COUNT(*) FROM bookings\").fetchone() == (2,)\n    assert free_rooms(conn, 550, 560) == [\"Boardroom\", \"Cellar\"] and free_rooms(conn, 660, 700) == [\"Attic\", \"Boardroom\", \"Cellar\"]")

### Exercise: Second chance: content negotiation (B5)

Write `negotiate(accept, available)`: `accept` is an `Accept` header such as `"text/html;q=0.8, application/json"` (each type may have a `q` weight; the default is 1); `available` lists the types the server can produce, in order of preference. Return the available type with the highest weight (`*/*` matches any type, and `text/*` any `text/` type; a type's weight is that of its most specific match), ties going to the earlier available type; `None` if nothing has a weight above 0 (the server would answer `406`). An empty header accepts anything.

In [ ]:
def negotiate(accept, available):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_negotiate():\n    \"weights, wildcards and preference\"\n    both = [\"application/json\", \"text/html\"]\n    assert negotiate(\"text/html;q=0.8, application/json\", both) == \"application/json\"\n    assert negotiate(\"text/html, application/json;q=0.5\", both) == \"text/html\"\n    assert negotiate(\"text/*;q=0.9, */*;q=0.1\", both) == \"text/html\"\n    assert negotiate(\"*/*\", both) == \"application/json\"\n    assert negotiate(\"\", both) == \"application/json\"\n    assert negotiate(\"image/png\", both) is None\n    assert negotiate(\"application/json;q=0, */*;q=0.5\", both) == \"text/html\", \"q=0 means not acceptable\"")

### Exercise: Second chance: one-time login codes (B6)

Write `LoginCodes(clock, make_code, ttl=300, max_attempts=3)`. `issue(user)` gets a code from `make_code()`, stores only its SHA-256 hex digest with an expiry, resets the user's failed attempts, and returns the code. `verify(user, code)` returns `True` (and uses the code up) if it matches (`hmac.compare_digest`) and hasn't expired (`now >= expiry` is expired); otherwise it counts a failed attempt and returns `False`; after `max_attempts` failures the code is locked, so even the right code returns `False` until a new one is issued.

In [ ]:
import hashlib
import hmac

class LoginCodes:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_codes():\n    \"hashed, limited, expiring, single use\"\n    class Clock:\n        t = 0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    codes = iter([\"111111\", \"222222\", \"333333\", \"444444\"])\n    lc = LoginCodes(clock, lambda: next(codes))\n    c = lc.issue(\"ada\")\n    assert c not in repr(vars(lc))\n    assert not lc.verify(\"ada\", \"999999\") and lc.verify(\"ada\", c) and not lc.verify(\"ada\", c)\n    c = lc.issue(\"ada\")\n    assert [lc.verify(\"ada\", \"000000\") for _ in range(3)] == [False, False, False]\n    assert not lc.verify(\"ada\", c), \"locked after three failures\"\n    c = lc.issue(\"ada\")\n    assert lc.verify(\"ada\", c), \"a new code unlocks\"\n    c = lc.issue(\"bo\")\n    clock.t = 300\n    assert not lc.verify(\"bo\", c) and not lc.verify(\"nobody\", \"1\")\n    assert \"compare_digest\" in source")

### Exercise: Second chance: the top customer in each city (B7)

Write `top_per_city(orders, customers)`: join orders (columns `customer_id`, `total`) to customers (`id`, `name`, `city`), keeping every order and refusing duplicated customer ids, then return a dict from each city (sorted) to `(name, spent)` for its customer with the highest total spending (rounded to 2 places; ties to the alphabetically first name).

In [ ]:
import pandas as pd

def top_per_city(orders, customers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_top():\n    \"merge, group, pick\"\n    customers = pd.DataFrame({\"id\": [1, 2, 3, 4], \"name\": [\"Ada\", \"Bo\", \"Cy\", \"Dee\"], \"city\": [\"York\", \"York\", \"Hull\", \"Hull\"]})\n    orders = pd.DataFrame({\"customer_id\": [1, 2, 2, 3, 4, 4, 1], \"total\": [30.0, 20.0, 15.5, 8.0, 4.0, 4.0, 5.5]})\n    assert top_per_city(orders, customers) == {\"Hull\": (\"Cy\", 8.0), \"York\": (\"Ada\", 35.5)}\n    tied = pd.DataFrame({\"customer_id\": [3, 4], \"total\": [5.0, 5.0]})\n    assert top_per_city(tied, customers) == {\"Hull\": (\"Cy\", 5.0)}\n    try:\n        top_per_city(orders, pd.concat([customers, customers.head(1)]))\n    except Exception:\n        pass\n    else:\n        raise AssertionError(\"duplicate ids must raise\")")

### Exercise: Second chance: choosing a tree's depth (B8)

Write `choose_depth(X, y, depths, folds=5)` like B8's `choose_k`, but for `DecisionTreeClassifier(max_depth=d, random_state=0)` (no scaling needed): your own consecutive folds (sizes differing by at most one, larger first), the mean accuracy per depth (3 places), and the best depth (ties: the **shallowest**, the simplest model). Return `(best, scores)`.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

def choose_depth(X, y, depths, folds=5):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from sklearn.datasets import make_classification\nfrom sklearn.model_selection import KFold, cross_val_score\nfrom sklearn.tree import DecisionTreeClassifier\n\ndef test_depth():\n    \"the same as KFold\"\n    X, y = make_classification(n_samples=200, n_features=5, n_informative=3, random_state=4)\n    best, scores = choose_depth(X, y, [1, 2, 3, 5, 8])\n    for d in (1, 5):\n        expected = cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), X, y, cv=KFold(5)).mean()\n        assert scores[d] == round(float(expected), 3)\n    assert best == min(d for d in scores if scores[d] == max(scores.values()))")

### Exercise: Second chance: managing API keys (C)

Write `KeyStore(make_key)`. `create(service)` gets a key from `make_key()`, stores only its SHA-256 digest for that service (a service has one key; creating again raises `ValueError("service exists")`), and returns the key. `check(key)` returns the service whose digest matches (compare each with `hmac.compare_digest`) or `None`. `rotate(service)` replaces the service's key with a new one and returns it, so the old key stops working at once; `revoke(service)` removes it. Both raise `KeyError` for an unknown service. `services()` lists the services, sorted.

In [ ]:
import hashlib
import hmac

class KeyStore:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_keys():\n    \"hashed, rotatable, revocable\"\n    keys = iter([\"k1-aaaa\", \"k2-bbbb\", \"k3-cccc\", \"k4-dddd\"])\n    store = KeyStore(lambda: next(keys))\n    shop = store.create(\"shop\")\n    blog = store.create(\"blog\")\n    assert shop not in repr(vars(store)) and store.check(shop) == \"shop\" and store.check(blog) == \"blog\"\n    try:\n        store.create(\"shop\")\n    except ValueError as e:\n        assert str(e) == \"service exists\"\n    new = store.rotate(\"shop\")\n    assert store.check(shop) is None and store.check(new) == \"shop\"\n    store.revoke(\"blog\")\n    assert store.check(blog) is None and store.services() == [\"shop\"]\n    for op in (store.rotate, store.revoke):\n        try:\n            op(\"nope\")\n        except KeyError:\n            pass\n        else:\n            raise AssertionError(\"unknown service\")\n    assert \"compare_digest\" in source")

## Your score

Write down your total before reading on. Then, as you go through each problem, note **why** you lost any marks:

1. **Didn't know it**: a gap in what you've learned. Revise the topic.
2. **Knew it, but couldn't use it**: you'd recognise the idea in a lesson but couldn't write it from scratch. Redo that week's exercises without looking.
3. **Edge case**: the main case worked, but a boundary value, an invalid input or a second call failed.
4. **Ran out of time**: the plan, not the knowledge, was the problem.

| Score | What it means |
|---|---|
| 144 or more (80%) | Distinction. You're well prepared for Phase 7. |
| 108 to 143 (60%) | Pass. Revise the topics where you lost marks during the rest week. |
| under 108 | Not yet. Spend the rest week on the topics below, then take the exam again. What matters is the second attempt. |

## Part A

| Questions | The idea | Revise |
|---|---|---|
| 1 | parametrize runs one test per case | [Week 72, Day 1](w072d1.html) |
| 2 | coverage shows what ran, not what's checked | [Week 72, Day 4](w072d4.html) |
| 3 | inject dependencies so tests can pass a mock | [Week 72, Day 2](w072d2.html) |
| 4 | libraries declare compatible ranges | [Week 73, Day 5](w073d5.html) |
| 5 | MAJOR versions mark breaking changes | [Week 73, Day 4](w073d4.html) |
| 6 | LEFT JOIN keeps every left row | [Week 75, Day 2](w075d2.html) |
| 7, 8 | transactions and indexes | [Week 75, Day 5](w075d5.html) |
| 9, 10 | idempotent methods and 429 | [Week 76, Day 2](w076d2.html) |
| 11 | slow, salted password hashes | [Week 77, Day 4](w077d4.html) |
| 12 | escaping stops cross-site scripting | [Week 79, Day 2](w079d2.html) |
| 13 | locks stop scheduled runs overlapping | [Week 78, Day 3](w078d3.html) |
| 14 | Little's law | [Week 81, Day 1](w081d1.html) |
| 15 | majorities overlap | [Week 82, Day 1](w082d1.html) |
| 16 | circuit breakers | [Week 82, Day 3](w082d3.html) |
| 17 | joins can multiply rows | [Week 83, Day 3](w083d3.html) |
| 18 | histograms show distributions | [Week 83, Day 4](w083d4.html) |
| 19 | precision is TP / (TP + FP) | [Week 84, Day 3](w084d3.html) |
| 20 | backpropagation reuses the forward pass | [Week 84, Day 4](w084d4.html) |

Question 2 catches people who treat coverage as a quality score: a test that calls every line but asserts nothing has full coverage and catches nothing. Question 17 is the classic silent data bug: a merge on a key that's duplicated on **both** sides multiplies rows, and every total after it is wrong without any error. That's why B7 asked for `validate="many_to_one"`.

## Part B

Each model solution runs, so try it. The notes give the key decision and the most common mistakes.

### B1: Tests for shipping costs

```python
import pytest
from shipping import cost

@pytest.mark.parametrize("weight, express, expected", [
    (1, False, 3.50), (2, False, 3.50), (2.1, False, 7.00), (10, False, 7.00),
    (10.2, False, 8.00), (13, False, 10.00), (2, True, 7.00),
])
def test_cost(weight, express, expected):
    assert cost(weight, express) == pytest.approx(expected)

@pytest.mark.parametrize("weight", [0, -1])
def test_rejects(weight):
    with pytest.raises(ValueError):
        cost(weight)
```

**The key decision**: test every **boundary** on both sides (2 and 2.1, 10 and 10.2) and one value well inside each band; boundaries are where `<` and `<=` mutants live. Use `pytest.approx` for prices. **Common mistakes**: testing only "typical" weights (1, 5, 20), which leaves the boundary mutants alive; forgetting that `0` must raise as well as negatives; checking `express` only on one band.

### B2: Predict

`p` is `"2.0.0"`: as tuples, `(2, 0, 0) < (9, 1, 0) < (10, 0, 0)`. `q` is `405`: the path exists but the method doesn't. `r` is `2` (York and Bath have more than one row). `s` is `12 / (12 + 4) = 0.75`; false positives don't affect recall. `t` is `99.99`: both replicas must fail together, `1 − 0.01² = 0.9999`.

### B3: Resolving dependencies

**The key decision**: collect **all** the clauses for each package before choosing, then take the maximum by **parsed** version. **Common mistakes**: comparing version strings (`"2.9.0" > "2.31.0"` as text), resolving each requirement separately so a later one overrides an earlier one, and `max` of the strings instead of `max(..., key=parse)`.

### B4: Signing up for events

**The key decision**: do the check and the insert (or the delete and the promotion) inside **one** `with conn:` block, so the capacity rule can't be broken between the two steps and a failure leaves nothing half done. `SUM(s.status = 'booked')` counts matches because a comparison is 0 or 1 in SQLite; the `LEFT JOIN` and `COALESCE` keep events with nobody signed up. **Common mistakes**: an inner join that drops empty events, promoting the most **recent** waiter instead of the earliest (`ORDER BY id`), and promoting someone when a **waiting** person cancels.

### B5: Conditional requests

**The key decision**: lower-case the request's header names once, then look up `if-none-match`. The ETag is computed from the **bytes** of the body, and so is `Content-Length`. **Common mistakes**: a case-sensitive header lookup; `len(body)` for the length (wrong for "é" and "£"); a body in the `HEAD` or `304` response; checking the method before the path, which gives `405` for pages that don't exist.

### B6: Password-reset tokens

**The key decision**: store `sha256(token)`, never the token, so a leaked database can't be used to reset anyone's password; delete the entry when it's redeemed (single use) **and** when a new one is issued (keying by user does that). **Common mistakes**: storing the token itself; comparing with `==`; letting an expired token through because the check is `>` instead of `>=`; leaving a used token valid.

### B7: Sales by category

**The key decision**: `merge(..., how="left", validate="many_to_one")` keeps every sale and refuses a products table with a repeated product, which would silently double revenue; `pivot_table(..., fill_value=0)` gives the zeros. **Common mistakes**: an inner join (losing sales for unknown products without noticing), forgetting `fill_value`, and taking the best seller by **quantity** instead of revenue (the Notebook sells most units in stationery, but Pens earn most).

### B8: Choosing k honestly

**The key decision**: a **new** pipeline for every fold, so the scaler is fitted only on that fold's training part, and the test set is touched only once, after `k` is chosen. Consecutive folds with the larger ones first match `KFold` exactly. **Common mistakes**: scaling all of `X` before the folds (a leak), choosing `k` by its **test** score, ties broken towards the larger `k`, and precision dividing by actual positives (that's recall).

## Part C

**The decisions that matter**:

- **Authenticate first**, before routing, so unknown paths don't reveal what exists to someone without a key. Compare against every key with `compare_digest`.
- **Validate the whole batch before storing any of it**: one bad event rejects the request and nothing is written, so a client can safely fix and resend. Note `isinstance(True, int)` is `True` in Python, so booleans need excluding explicitly.
- **Scope every query by service**, through a `?` parameter: one service's data never leaks into another's report.
- **Keep the arithmetic in one place**: the nearest-rank percentile and the budget formula are the same as Week 81 and Week 82, now behind an API.

**Common mistakes**: returning `404` before checking the key, storing the valid events of a partly invalid batch, forgetting the time window, a `p95` over all endpoints when `endpoint` was given, `ZeroDivisionError` when there are no events, and the `status` thresholds off by one (`0.5` exactly is `"ok"`).

## Second chances

Each exercise practises the skill behind one exam problem, on a fresh problem. Do them without looking at the model solutions above.

## What's next

**Your revision plan.** For each problem where you lost marks, write one line: the problem, the kind of mistake, and the day you'll revisit. Do those days during the rest week. If you're retaking the exam, do it after the rest week, without looking at this page again first.

**Phase 7: mastery.** After a week's rest, the last phase is about doing the work for real. It starts with a six-week **capstone**: a project of your own choosing, taken from a written specification through design review, build sprints with tests and code review, hardening (performance, security, logging, documentation, user testing) to a release, a write-up and a portfolio page. Then come **open-source contributions**, including to CPython itself: reading a large codebase, building it from source, finding, reproducing and fixing an issue, and responding to review. The phase ends with interview practice, a final exam and your portfolio. Everything in Phase 6 is in daily use there: tests and CI from the first day, packaging and versions for the release, a database and an API for most capstones, security review before shipping, and measurement to know whether it works.

You've finished Phase 6. Fourteen weeks ago a program was something that ran on your computer; since then you've tested it properly, packaged and versioned it, kept its quality up with review and CI, stored its data in a database, put it on the web behind an API with authentication, automated its chores, defended it against attack, built a full project, designed systems that grow and survive failure, analysed data and charted it honestly, and taught a machine to learn from examples. Take the rest week: you've earned it.